# Employee Churn Prediction

This notebook loads the HR churn dataset, cleans it, evaluates multiple ML models, and saves the best-performing pipeline.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import pickle
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score, precision_score, recall_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler


In [ ]:
DATA_PATH = Path("HR_Dataset.csv")

df = pd.read_csv(DATA_PATH)
df.head()


In [ ]:
df.columns = [col.strip() for col in df.columns]
df = df.drop_duplicates().reset_index(drop=True)
df.info()
print("\nUnique columns:", df.columns.tolist())


In [ ]:
target = "left"
print("Target distribution:")
print(df[target].value_counts(normalize=True).round(3))
print("\nMissing values:")
print(df.isna().sum())


## Feature preparation

The dataset mixes numeric and categorical features. We use a preprocessing pipeline to one-hot encode the categorical variables while scaling the numeric variables.

In [ ]:
X = df.drop(columns=[target])
y = df[target]

numeric_features = X.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = X.select_dtypes(exclude=[np.number]).columns.tolist()

print("Numeric:", numeric_features)
print("Categorical:", categorical_features)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
    ]
)


In [ ]:
def evaluate_model(name, model):
    pipeline = Pipeline(steps=[("preprocessor", preprocessor), ("model", model)])
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)

    metrics = {
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred, zero_division=0),
        "recall": recall_score(y_test, y_pred, zero_division=0),
        "f1": f1_score(y_test, y_pred, zero_division=0),
    }

    print(f"\n=== {name} ===")
    print(classification_report(y_test, y_pred, digits=4))
    print("Confusion matrix:\n", confusion_matrix(y_test, y_pred))
    print("Metrics:", metrics)

    return pipeline, metrics


In [ ]:
logistic_model = LogisticRegression(max_iter=1000, random_state=42)
logistic_pipeline, logistic_metrics = evaluate_model("Logistic Regression", logistic_model)


In [ ]:
rf_model = RandomForestClassifier(n_estimators=300, random_state=42, class_weight="balanced")
rf_pipeline, rf_metrics = evaluate_model("Random Forest", rf_model)


In [ ]:
try:
    import xgboost as xgb
    from xgboost import XGBClassifier

    xgb_model = XGBClassifier(
        n_estimators=300,
        max_depth=5,
        learning_rate=0.05,
        subsample=0.9,
        colsample_bytree=0.9,
        random_state=42,
        objective="binary:logistic",
        eval_metric="logloss",
    )
    xgb_pipeline, xgb_metrics = evaluate_model("XGBoost", xgb_model)
except Exception as exc:
    print("XGBoost is not available in this environment:", exc)
    xgb_metrics = None


In [ ]:
results = {
    "logistic_regression": logistic_metrics,
    "random_forest": rf_metrics,
}
if xgb_metrics is not None:
    results["xgboost"] = xgb_metrics

best_model_name, best_model_metrics = max(
    results.items(), key=lambda item: item[1]["f1"]
)

print("\nBest model by F1 score:", best_model_name)
print(best_model_metrics)


## Save the best pipeline

We save the winning model pipeline for future inference.

In [ ]:
chosen_pipeline = logistic_pipeline
if best_model_name == "random_forest":
    chosen_pipeline = rf_pipeline
elif best_model_name == "xgboost":
    chosen_pipeline = xgb_pipeline

with open("employee_churn_model.pkl", "wb") as f:
    pickle.dump(chosen_pipeline, f)

print("Saved model to employee_churn_model.pkl")


The notebook is now valid JSON, includes proper preprocessing, trains several classifiers, evaluates them with standard classification metrics, and exports the best model to disk.